# Recommendation / Message Personalization

use hitorical interaction logs instead of environment (offline reinforcement learning / contextual bandit style)

system ex.
- push notification
- email campaign
- message queue prioritization
- ad recommendation

Environment -> user interaction (e.g., click, ignore, etc.)

### Step 1
state = user feature -> ex. state = (segment, last_click)

### Step 2: Q-table

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
import pandas as pd

data = {
    "user_id": [101,101,202,203],
    "segment": ["new","new","vip","new"],
    "last_click": [0,0,1,1],
    "action": ["promo","reminder","upsell","education"],
    "reward": [0,1,5,0]
}

df = pd.DataFrame(data)
df

,user_id,segment,last_click,action,reward
0,101,new,0,promo,0
1,101,new,0,reminder,1
2,202,vip,1,upsell,5
3,203,new,1,education,0


In [3]:
actions = ["promo", "reminder", "education", "upsell"]

states = [
    ("new", 0),
    ("new", 1),
    ("vip", 0),
    ("vip", 1)
]

state_to_idx = {s:i for i, s in enumerate(states)}
action_to_idx = {a:i for i, a in enumerate(actions)}

Q = np.zeros((len(states), len(actions)))
display(Q)

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### Step 3: Train from Historical Logs

non environment -> use (state, action, reward, next_state)

In [6]:
alpha = 0.1
gamma = 0.9

for i in range(len(df)-1):
    
    s = (df.iloc[i].segment, df.iloc[i].last_click)
    a = df.iloc[i].action
    r = df.iloc[i].reward
    
    s_next = (df.iloc[i+1].segment, df.iloc[i+1].last_click)
    
    s_idx = state_to_idx[s]
    a_idx = action_to_idx[a]
    s_next_idx = state_to_idx[s_next]
    
    Q[s_idx, a_idx] += alpha * (
        r + gamma * np.max(Q[s_next_idx]) - Q[s_idx, a_idx]
    )

### Step 4: Inference

message queue -> choose message

In [7]:
def recommend(segment, last_click):
    s = (segment, last_click)
    s_idx = state_to_idx[s]
    
    best_action = np.argmax(Q[s_idx])
    return actions[best_action]

In [8]:
recommend("vip", 1)

'upsell'

### Step 5: Integrate with Message Queue

User event -> Feature store -> RL policy -> Choose message -> Send to queue